# XGBoost

Nesse notebook nós iremos treinar um modelo XGBoost em nosso dataset médico. O objetivo aqui é checar possíveis melhorias usando a abordagem de um modelo baseado árvores de decisão e boosting.

## Importando as Bibliotecas

Primeiro vamos carregar nossas bibliotecas.

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## Carregando o Dataset

Agora vamos carregar nosso dataset.

In [2]:
df = pd.read_parquet("../../data/processed/UCMF_fitted.parquet")

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 11705 entries, 0 to 12872
Data columns (total 30 columns):
 #   Column                  Non-Null Count  Dtype   
---  ------                  --------------  -----   
 0   peso                    9587 non-null   Float64 
 1   altura                  8078 non-null   Int64   
 2   imc                     7708 non-null   Int64   
 3   idade                   10865 non-null  Float64 
 4   pulsos                  11657 non-null  category
 5   pa_sistolica            5131 non-null   Int64   
 6   pa_diastolica           5121 non-null   Int64   
 7   ppa                     10768 non-null  category
 8   patologia               11705 non-null  category
 9   b2                      11674 non-null  category
 10  sopro                   11682 non-null  category
 11  fc                      10986 non-null  Int64   
 12  hda1                    8565 non-null   category
 13  hda2                    11705 non-null  category
 14  sexo                    11701 non-null

## Seleção de Modelo

Para selecionar o melhor modelo de XGBoost vamos otimizar os hiperparâmetros usando a biblioteca `Optuna` que faz Otimização Bayesiana. A escolha do `Optuna` se dá pelo fato de nós otimizarmos muitos hiperparâmetros com diversos valores e essa biblioteca é perfeita para tarefas custosas como essa. Para isso vamos primeiro montar nosso pré-processador.

In [15]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import (
    SimpleImputer,
)
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder,
)
from sklearn.model_selection import (
    StratifiedKFold,
    cross_val_score,
    cross_validate,
    train_test_split,
)

def create_preprocessor(X):
    numeric_columns = X.select_dtypes(include="number").columns.to_list()
    categorical_columns = X.select_dtypes(include="category").columns.to_list()

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric_features", numeric_pipeline, numeric_columns),
            ("categorical_features", categorical_pipeline, categorical_columns),
        ],
        remainder="passthrough"
    )

    return preprocessor

Agora vamos criar a função objetivo que o `Optuna` irá otimizar. Internamente os parâmetros dessa função serão nossos hiperparâmetros e a imagem dela será o $\text{ROC-AUC}$, portanto o `Optuna` tentará achar os parâmetros (hiperparâmetros) que maximizem a imagem ($\text{ROC-AUC}$).

In [10]:
from xgboost import XGBClassifier

random_state = 42

cross_validator = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state,
)

def create_objetive(X, y):
    preprocessor = create_preprocessor(X)

    def objective(trial):
        model = XGBClassifier(
            objective="binary:logistic",
            eval_metric="aucpr",
            random_state=random_state,
            tree_method="hist",
            n_estimators=trial.suggest_int("n_estimators", 200, 2000),
            learning_rate=trial.suggest_float("learning_rate", 0.001, 0.3, log=True),
            max_depth=trial.suggest_int("max_depth", 3, 10),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 20),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            gamma=trial.suggest_float("gamma", 0, 10),
            reg_alpha=trial.suggest_float("reg_alpha", 0.000001, 10, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 0.000001, 10, log=True),
            scale_pos_weight = trial.suggest_float("scale_pos_weight", 1.0, 3.0)
        )

        pipeline = Pipeline([
            ("preprocessor", preprocessor),
            ("model", model)
        ])

        scores = cross_val_score(
            pipeline,
            X,
            y,
            cv=cross_validator,
            scoring="roc_auc",
            n_jobs=-1
        )

        return scores.mean()
    
    return objective

Então vamos criar nossa função objetivo com o dataset inteiro.

In [11]:
label = LabelEncoder()

X = df.drop(["patologia"], axis="columns")
y = label.fit_transform(df["patologia"])

objective = create_objetive(X, y)

E agora vamos otimizar os hiperparâmetros.

In [13]:
import optuna

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.PercentilePruner(25.0, n_startup_trials=5)
)

study.optimize(
    objective,
    n_trials=100,
    show_progress_bar=True,
    n_jobs=-1
)

[I 2026-06-13 16:05:34,757] A new study created in memory with name: no-name-5275d79a-1a70-4eab-b77f-9bd3a51ecee3
  0%|          | 0/100 [00:00<?, ?it/s]/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 0. Best value: 0.948643:   1%|          | 1/100 [00:03<05:48,  3.52s/it]

[I 2026-06-13 16:05:38,258] Trial 0 finished with value: 0.94864266242617 and parameters: {'n_estimators': 949, 'learning_rate': 0.05321007620778028, 'max_depth': 10, 'min_child_weight': 18, 'subsample': 0.8743963523555871, 'colsample_bytree': 0.7576230014207923, 'gamma': 1.277870614206822, 'reg_alpha': 1.3694358243985647, 'reg_lambda': 8.326823645483015e-06, 'scale_pos_weight': 2.173543102664842}. Best is trial 0 with value: 0.94864266242617.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 0. Best value: 0.948643:   2%|▏         | 2/100 [00:05<04:05,  2.50s/it]

[I 2026-06-13 16:05:40,073] Trial 1 finished with value: 0.9469814856883236 and parameters: {'n_estimators': 820, 'learning_rate': 0.1531772357916516, 'max_depth': 8, 'min_child_weight': 6, 'subsample': 0.686997229982703, 'colsample_bytree': 0.7762899894473392, 'gamma': 5.022035536471617, 'reg_alpha': 0.16573434524146435, 'reg_lambda': 0.05856587892860511, 'scale_pos_weight': 2.5821537636004344}. Best is trial 0 with value: 0.94864266242617.


Best trial: 0. Best value: 0.948643:   3%|▎         | 3/100 [00:05<02:25,  1.50s/it]

...

[I 2026-06-13 16:11:20,125] Trial 97 finished with value: 0.9518420155909993 and parameters: {'n_estimators': 1624, 'learning_rate': 0.002723807601608579, 'max_depth': 10, 'min_child_weight': 2, 'subsample': 0.6330707976181978, 'colsample_bytree': 0.7679406192969278, 'gamma': 1.2284675297072805, 'reg_alpha': 0.18570962018536982, 'reg_lambda': 1.325647872090044e-05, 'scale_pos_weight': 1.5787072129636197}. Best is trial 94 with value: 0.9523997764174179.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 94. Best value: 0.9524:  99%|█████████▉| 99/100 [05:56<00:09,  9.25s/it]

[I 2026-06-13 16:11:31,531] Trial 98 finished with value: 0.9521672266194792 and parameters: {'n_estimators': 1865, 'learning_rate': 0.003173305817040089, 'max_depth': 10, 'min_child_weight': 2, 'subsample': 0.7024086792729005, 'colsample_bytree': 0.7012370003963071, 'gamma': 0.7581553131456964, 'reg_alpha': 0.14454215668254963, 'reg_lambda': 1.2645886737866735e-05, 'scale_pos_weight': 1.813592380385048}. Best is trial 94 with value: 0.9523997764174179.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 94. Best value: 0.9524: 100%|██████████| 100/100 [06:09<00:00,  3.69s/it]

[I 2026-06-13 16:11:44,154] Trial 99 finished with value: 0.952147424017791 and parameters: {'n_estimators': 1846, 'learning_rate': 0.0025494951946663266, 'max_depth': 10, 'min_child_weight': 2, 'subsample': 0.7024449075087218, 'colsample_bytree': 0.7696828430264564, 'gamma': 0.9037504811821429, 'reg_alpha': 0.15404692698863262, 'reg_lambda': 3.933255389191584e-05, 'scale_pos_weight': 1.8385495025889405}. Best is trial 94 with value: 0.9523997764174179.


Agora vamos montar o pipeline do nosso modelo ótimo.

In [14]:
params = study.best_params

model = XGBClassifier(
    **params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=random_state,
    tree_method="hist",
)

preprocessor = create_preprocessor(X)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

E vamos avaliar ele.

In [17]:
scores = cross_validate(
    pipeline,
    X,
    y,
    cv=cross_validator,
    scoring={
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    n_jobs=-1,
)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [5, 8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [27]:
for metric, values in scores.items():
    print(f"{metric:14} = {(100*values.mean()).round(2)}")

fit_time       = 1067.33
score_time     = 45.01
test_roc_auc   = 95.24
test_accuracy  = 92.87
test_precision = 95.05
test_recall    = 87.76
test_f1        = 91.26


Como podemos notar, as métricas são tão excelentes quanto nos modelos anteriores. Vamos então analisar a importância das nossas features.

In [38]:
pipeline.fit(X, y)

feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
feature_importances = pipeline.named_steps["model"].feature_importances_

importances = pd.DataFrame({
    "feature": feature_names,
    "importance": feature_importances,
}).sort_values(by="importance", ascending=False)

print(importances.to_string())

                                                          feature  importance
31                          categorical_features__sopro_sistolico    0.657263
26                                categorical_features__b2_normal    0.032950
52             categorical_features__motivo1_parecer_cardiologico    0.021862
25                         categorical_features__b2_hiperfonetica    0.016681
68                            categorical_features__motivo2_sopro    0.016661
57            categorical_features__motivo2_cardiopatia_congenica    0.016121
29                           categorical_features__sopro_continuo    0.010034
16                           categorical_features__pulsos_normais    0.006807
28                                 categorical_features__b2_unica    0.006796
27                                 categorical_features__b2_outro    0.006104
30                         categorical_features__sopro_diastolico    0.006033
50                         categorical_features__motivo1_check_u

Como podemos notar, a feature mais importante continua sendo `"sopro"` com valor `"sistolico"` e ela é disparada a feature mais importante.

Agora, diferente do que fizemos nos modelos anteriores, vamos refazer a otimização de hiperparametros mas sem a variável `"sopro"`. A motivação para isso é o fato de que modelos baseados em Boosting com o XGBoost, LightGBM e CatBoost são muito poderoso e talvez eles encontrem alguma relação entre as features sem `"sopro"` e o target que os outros modelos não encontraram.

In [29]:
objective_sem_sopro = create_objetive(X.drop("sopro", axis="columns"), y)

study = optuna.create_study(
    direction="maximize",
    pruner=optuna.pruners.PercentilePruner(25.0, n_startup_trials=5)
)

study.optimize(
    objective_sem_sopro,
    n_trials=100,
    show_progress_bar=True,
    n_jobs=-1
)

[I 2026-06-13 16:28:41,681] A new study created in memory with name: no-name-4c54104c-f463-4e00-b1ee-3df3a9c07f20
  0%|          | 0/100 [00:00<?, ?it/s]/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 2. Best value: 0.808122:   1%|          | 1/100 [00:04<07:17,  4.41s/it]

[I 2026-06-13 16:28:46,089] Trial 2 finished with value: 0.808121756856931 and parameters: {'n_estimators': 947, 'learning_rate': 0.05133564064399223, 'max_depth': 6, 'min_child_weight': 17, 'subsample': 0.6726689330451595, 'colsample_bytree': 0.9982749261336382, 'gamma': 7.655383897216025, 'reg_alpha': 0.013366866159396874, 'reg_lambda': 1.1689394382473777e-05, 'scale_pos_weight': 1.576491964116663}. Best is trial 2 with value: 0.808121756856931.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 3. Best value: 0.808395:   2%|▏         | 2/100 [00:10<08:49,  5.40s/it]

[I 2026-06-13 16:28:52,180] Trial 3 finished with value: 0.8083945096726369 and parameters: {'n_estimators': 1959, 'learning_rate': 0.09285514312573959, 'max_depth': 7, 'min_child_weight': 14, 'subsample': 0.8249645473219476, 'colsample_bytree': 0.8652697575559887, 'gamma': 9.358653112961772, 'reg_alpha': 0.004081113558767837, 'reg_lambda': 0.0041138096947447396, 'scale_pos_weight': 1.6921588644376313}. Best is trial 3 with value: 0.8083945096726369.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 3. Best value: 0.808395:   3%|▎         | 3/100 [00:13<07:09,  4.43s/it]

...

[I 2026-06-13 16:33:04,095] Trial 97 finished with value: 0.8088693237523614 and parameters: {'n_estimators': 1750, 'learning_rate': 0.06290577503452827, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8158152140226516, 'colsample_bytree': 0.6196191502028081, 'gamma': 4.279630605017196, 'reg_alpha': 0.008835339451940219, 'reg_lambda': 0.5355643250865622, 'scale_pos_weight': 1.9637189113752074}. Best is trial 57 with value: 0.8113764078528899.


/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
Best trial: 57. Best value: 0.811376:  99%|█████████▉| 99/100 [04:26<00:03,  3.14s/it]

[I 2026-06-13 16:33:08,173] Trial 98 finished with value: 0.8110594167722806 and parameters: {'n_estimators': 1740, 'learning_rate': 0.006954407666796061, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8213502410315787, 'colsample_bytree': 0.6514432913030896, 'gamma': 4.319998651952965, 'reg_alpha': 0.007730964377156366, 'reg_lambda': 1.1099789506227005, 'scale_pos_weight': 1.9717956749950338}. Best is trial 57 with value: 0.8113764078528899.


Best trial: 57. Best value: 0.811376: 100%|██████████| 100/100 [04:28<00:00,  2.68s/it]

[I 2026-06-13 16:33:09,707] Trial 99 finished with value: 0.8046562268346923 and parameters: {'n_estimators': 1759, 'learning_rate': 0.1631254591009625, 'max_depth': 6, 'min_child_weight': 11, 'subsample': 0.8919106141407036, 'colsample_bytree': 0.5883580155966659, 'gamma': 3.2602614891061723, 'reg_alpha': 0.008423678247301981, 'reg_lambda': 1.184994289341145, 'scale_pos_weight': 2.1523341327477286}. Best is trial 57 with value: 0.8113764078528899.


Vamos montar nosso pipeline.

In [ ]:
params_sem_sopro = study.best_params

model_sem_sopro = XGBClassifier(
    **params,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=random_state,
    tree_method="hist",
)

preprocessor_sem_sopro = create_preprocessor(X.drop("sopro", axis="columns"))

pipeline_sem_sopro = Pipeline(steps=[
    ("preprocessor", preprocessor_sem_sopro),
    ("model", model_sem_sopro)
])

E calcular as métricas de desempenho.

In [33]:
scores_sem_sopro = cross_validate(
    pipeline_sem_sopro,
    X.drop("sopro", axis="columns"),
    y,
    cv=cross_validator,
    scoring={
        "roc_auc": "roc_auc",
        "accuracy": "accuracy",
        "precision": "precision",
        "recall": "recall",
        "f1": "f1"
    },
    n_jobs=-1,
)

/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/home/gauloish/.dev/ds/ufg/mineração de dados/condicao-cardiaca/.venv/lib/python3.12/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 7] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


In [34]:
for metric, values in scores_sem_sopro.items():
    print(f"{metric:14} = {(100*values.mean()).round(2)}")

fit_time       = 1166.9
score_time     = 43.62
test_roc_auc   = 80.9
test_accuracy  = 73.79
test_precision = 68.46
test_recall    = 70.77
test_f1        = 69.59


Como podemos ver, as métricas sem a feature `"sopro"` continuam baixas como nos demais modelos, mas tivemos uma melhoria no recall.